# Feature Engineering & Preprocessing Pipeline — Titanic

## Internship Task 03

This notebook demonstrates a reusable, leak-free Scikit-Learn preprocessing pipeline on the Titanic survival dataset.

**Requirements covered:**
- Load and inspect a tabular dataset
- Perform train/test splitting before fitting transformations
- Engineer useful features
- Use `Pipeline` and `ColumnTransformer`
- Impute missing values
- Scale numerical variables
- One-hot encode categorical variables
- Train a tree-based model
- Rank transformed features by tree-based feature importance

The Titanic dataset is a standard classification dataset. The target is `survived` (0 = did not survive, 1 = survived).

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer

## 2. Load the Titanic dataset

This notebook first looks for a local CSV at `data/titanic.csv`. If it is not present, it loads the Titanic dataset from OpenML. This makes the notebook convenient to run in VS Code while still keeping the dataset source explicit.

In [ ]:
LOCAL_PATH = "data/titanic.csv"

if __import__('os').path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f"Loaded local dataset: {LOCAL_PATH}")
else:
    from sklearn.datasets import fetch_openml
    df = fetch_openml("titanic", version=1, as_frame=True).frame
    print("Loaded Titanic dataset from OpenML")

df.head()

## 3. Understand the dataset

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
        
display(df.head())

In [ ]:
df.info()

## 4. Data quality checks

Before building the model, inspect missing values, duplicate rows, and the target distribution.

In [ ]:
print("Missing values by column:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["survived"].value_counts(dropna=False).to_frame("count"))

## 5. Select modelling features and target

We deliberately use a compact set of passenger attributes that are available before the survival outcome.

We **do not** use columns such as `alive`, `boat`, or `body` when they are present in the source data because they reveal information about the outcome after the event. Using such columns would create target leakage.

We also exclude identifiers/free-text fields such as passenger name and ticket number from this baseline.

In [ ]:
feature_columns = [
    "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"
]

missing_expected = [c for c in feature_columns + ["survived"] if c not in df.columns]
if missing_expected:
    raise ValueError(f"Expected columns not found: {missing_expected}")

X = df[feature_columns].copy()
y = pd.to_numeric(df["survived"], errors="coerce").astype(int)

print("Features:", X.columns.tolist())
print("Target: survived")

## 6. Train/test split — before fitting transformations

The split happens **before** imputation, scaling, encoding, or feature-selection statistics are learned. This prevents the test set from influencing the preprocessing parameters.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 7. Feature engineering

Two simple features are derived only from information already present for each passenger:

- `family_size = sibsp + parch + 1`
- `is_alone = 1` when family size is 1, otherwise `0`

Because these features are calculated row-by-row without using `survived`, they do not directly leak the target.

In [ ]:
def add_titanic_features(data):
    data = data.copy()
    data["family_size"] = data["sibsp"] + data["parch"] + 1
    data["is_alone"] = (data["family_size"] == 1).astype(int)
    return data

X_train_fe = add_titanic_features(X_train)
display(X_train_fe.head())

## 8. Define numerical and categorical preprocessing

Numerical columns use median imputation followed by standardization. Categorical columns use most-frequent imputation followed by one-hot encoding.

This is the same mixed-type strategy demonstrated in Scikit-Learn's `ColumnTransformer` examples. citeturn0search7

In [ ]:
numeric_features = [
    "age", "sibsp", "parch", "fare", "family_size", "is_alone"
]

categorical_features = ["pclass", "sex", "embarked"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

## 9. Combine preprocessing with ColumnTransformer

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

## 10. Build the complete reusable ML Pipeline

The feature-engineering step, preprocessing step, and model are chained together. This means the same sequence is applied consistently during training and prediction.

In [ ]:
feature_engineering = FunctionTransformer(
    add_titanic_features,
    validate=False
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

pipeline = Pipeline(steps=[
    ("feature_engineering", feature_engineering),
    ("preprocessor", preprocessor),
    ("model", model)
])

## 11. Train the pipeline

In [ ]:
pipeline.fit(X_train, y_train)
print("Pipeline training completed successfully.")

## 12. Evaluate on the untouched test set

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
        
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=3))
        
        
        

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:")
print(cm)

## 13. Inspect the transformed feature names

One-hot encoding creates multiple output columns from each categorical feature. `get_feature_names_out()` lets us map model coefficients/importances back to readable transformed feature names.

In [ ]:
fitted_preprocessor = pipeline.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("First transformed features:")
print(feature_names[:20])

## 14. Tree-based feature importance

The Random Forest provides an impurity-based feature-importance score for each transformed feature. These scores describe how much each feature contributed to the trees' split decisions.

Important: feature importance is not proof of causality. It should be interpreted as model-specific evidence, not as a statement that a feature causes survival.

In [ ]:
rf_model = pipeline.named_steps["model"]
importances = rf_model.feature_importances_

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(15))

In [ ]:
top_n = 15
plot_df = importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Random Forest feature importance")
plt.ylabel("Transformed feature")
plt.title("Top transformed feature importances")
plt.tight_layout()
plt.show()

## 15. Optional: permutation importance on the held-out test set

Permutation importance evaluates the effect of shuffling a feature on the fitted pipeline's test performance. Unlike impurity-based importance, it can be evaluated directly against the held-out data. Here it is applied to the original input columns.

In [ ]:
perm = permutation_importance(
    pipeline,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"
)

perm_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "mean_importance": perm.importances_mean,
        "std": perm.importances_std
    })
    .sort_values("mean_importance", ascending=False)
    .reset_index(drop=True)
)

display(perm_df)

## 16. Leakage and reproducibility checklist

- [x] Train/test split occurs before fitting preprocessing.
- [x] Imputation parameters are learned only from training data through the pipeline.
- [x] Scaling parameters are learned only from training data through the pipeline.
- [x] One-hot encoding is fitted only on training data.
- [x] `handle_unknown="ignore"` prevents unseen test categories from breaking prediction.
- [x] Target-derived fields such as `alive`, `boat`, and `body` are excluded from the baseline features.
- [x] Feature engineering uses only passenger attributes available in the selected input data.
- [x] A fixed `random_state` makes the split and Random Forest reproducible.

## Conclusion

This notebook satisfies the core requirements of the internship task by demonstrating feature engineering, mixed-type preprocessing, a reusable Scikit-Learn `Pipeline`, a `ColumnTransformer`, leakage-aware train/test separation, and tree-based feature importance.